# 06 · Fine-tuning de instruções: virando assistente — *Rafael*

**Entra:** GPT-2 124M pré-treinado (04). **Sai:** um modelo que responde instruções.

Perguntas:
1. Como um modelo que só "continua texto" vira um assistente?
2. Onde eu consigo dados de fine-tuning? Como construo do zero?

In [1]:
import json
import os
import time
from functools import partial

import tiktoken
import torch
from torch.utils.data import DataLoader

from aula import REPO_DIR, CHECKPOINTS, device, load_gpt2
from llms_from_scratch.ch05 import generate, text_to_token_ids, token_ids_to_text, train_model_simple
from llms_from_scratch.ch07 import format_input, InstructionDataset, custom_collate_fn

CARREGAR_CHECKPOINT = True  # 🔧 True = usa o assistente salvo (se existir)
N_EXEMPLOS_TREINO = 400      # 🔧 subconjunto para caber ao vivo (o livro usa ~935 e o GPT-2 355M)
bpe = tiktoken.get_encoding("gpt2")

## 1. Os dados: pares instrução → resposta

In [2]:
with open(os.path.join(REPO_DIR, "ch07", "01_main-chapter-code", "instruction-data.json")) as f:
    dados = json.load(f)
print(len(dados), "exemplos\n")
print(json.dumps(dados[50], indent=2, ensure_ascii=False))

1100 exemplos

{
  "instruction": "Identify the correct spelling of the following word.",
  "input": "Ocassion",
  "output": "The correct spelling is 'Occasion.'"
}


Formatamos no estilo **Alpaca** — o modelo aprende o "molde" e onde a resposta começa:

In [3]:
print(format_input(dados[50]) + f"\n\n### Response:\n{dados[50]['output']}")

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [4]:
n_tr, n_te = int(len(dados) * 0.85), int(len(dados) * 0.10)
treino, teste, val = dados[:n_tr], dados[n_tr:n_tr + n_te], dados[n_tr + n_te:]
treino = treino[:N_EXEMPLOS_TREINO]
print(len(treino), "treino |", len(val), "validação |", len(teste), "teste")

400 treino | 55 validação | 110 teste


## 2. Batches: padding e o `-100`
Frases têm tamanhos diferentes → completamos com `<|endoftext|>` (50256). No alvo, o padding extra vira **-100**,
que o `cross_entropy` ignora: o modelo não é punido por prever "padding".

In [5]:
collate = partial(custom_collate_fn, device=device, allowed_max_length=1024)
x, y = collate([[1, 2, 3, 4, 5], [6, 7], [8, 9, 10]])
print("entradas:\n", x.cpu(), "\nalvos:\n", y.cpu())

entradas:
 tensor([[    1,     2,     3,     4,     5],
        [    6,     7, 50256, 50256, 50256],
        [    8,     9,    10, 50256, 50256]]) 
alvos:
 tensor([[    2,     3,     4,     5, 50256],
        [    7, 50256,  -100,  -100,  -100],
        [    9,    10, 50256,  -100,  -100]])


In [6]:
torch.manual_seed(123)
train_loader = DataLoader(InstructionDataset(treino, bpe), batch_size=8, collate_fn=collate, shuffle=True, drop_last=True)
val_loader = DataLoader(InstructionDataset(val, bpe), batch_size=8, collate_fn=collate)

## 3. Antes do fine-tuning

In [7]:
gpt2, cfg = load_gpt2("124M")
gpt2.to(device)

def responder(model, entrada, max_new_tokens=60):
    prompt = format_input(entrada)
    ids = generate(model, text_to_token_ids(prompt, bpe).to(device), max_new_tokens=max_new_tokens,
                   context_size=cfg["context_length"], eos_id=50256)
    return token_ids_to_text(ids, bpe)[len(prompt):].replace("### Response:", "").strip()

for e in teste[:3]:
    print(f"▶ {e['instruction']} {e['input']}\n  modelo: {responder(gpt2, e)!r}\n")

▶ Rewrite the sentence using a simile. The car is very fast.
  modelo: '### Output:\n\nThe car is very fast.\n\n### Error:\n\nThe car is very fast.\n\n### Error:\n\nThe car is very fast.\n\n### Error:\n\nThe car is very fast.\n\n### Error:\n\nThe'



▶ What type of cloud is typically associated with thunderstorms? 
  modelo: 'Thunderstorms are the most common type of thunderstorm. They are usually caused by lightning, thunderstorms, or other natural phenomena.\n\nThunderstorms are caused by a combination of natural phenomena, such as lightning, thunderstorms, or other natural phenomena.\n\nThunderstorms are caused by'



▶ Name the author of 'Pride and Prejudice'. 
  modelo: "### Description:\n\nThe author of 'Pride and Prejudice' is a young man who has been a member of the Church for over a century. He is a member of the Church of England and has been a member of the Church for over a century. He is a"



## 4. Fine-tuning (ou checkpoint)
É o **mesmo loop do pré-treino** (`train_model_simple`) — só mudaram os dados.

In [8]:
ckpt = os.path.join(CHECKPOINTS, "assistente_124M.pth")
if CARREGAR_CHECKPOINT and os.path.exists(ckpt):
    gpt2.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    print("checkpoint carregado:", ckpt)
else:
    torch.manual_seed(123)
    t0 = time.time()
    optimizer = torch.optim.AdamW(gpt2.parameters(), lr=5e-5, weight_decay=0.1)
    train_model_simple(gpt2, train_loader, val_loader, optimizer, device, num_epochs=2,
                       eval_freq=10, eval_iter=5, start_context=format_input(val[0]), tokenizer=bpe)
    print(f"{(time.time() - t0) / 60:.1f} min")
    # bfloat16 = metade do tamanho (~330 MB); load_state_dict converte de volta para float32
    torch.save({k: v.to(torch.bfloat16) for k, v in gpt2.state_dict().items()}, ckpt + ".part")
    os.replace(ckpt + ".part", ckpt)

Ep 1 (Step 000000): Train loss 3.022, Val loss 3.057


Ep 1 (Step 000010): Train loss 1.163, Val loss 1.216


Ep 1 (Step 000020): Train loss 0.974, Val loss 1.062


Ep 1 (Step 000030): Train loss 0.880, Val loss 1.010


Ep 1 (Step 000040): Train loss 0.771, Val loss 0.943


...

Ep 2 (Step 000060): Train loss 0.677, Val loss 0.886


Ep 2 (Step 000070): Train loss 0.559, Val loss 0.892


Ep 2 (Step 000080): Train loss 0.665, Val loss 0.858


Ep 2 (Step 000090): Train loss 0.527, Val loss 0.861


Below is an instruction that describes a task. Write a response that appropriately completes the request.  ### Instruction: Convert the active sentence to passive: 'The chef cooks the meal every day.'  ### Response: The chef cooks the meal every day.<|endoftext|>The following is an instruction that describes a task. Write a response that appropriately completes the request.  ### Input: What is the chemical symbol for carbon?  
1.1 min


## 5. Depois do fine-tuning

In [9]:
gpt2.eval()
torch.manual_seed(123)
for e in teste[:6]:
    print(f"▶ {e['instruction']} {e['input']}\n  esperado: {e['output']}\n  modelo:   {responder(gpt2, e)}\n")

▶ Rewrite the sentence using a simile. The car is very fast.
  esperado: The car is as fast as lightning.
  modelo:   The car is very fast.



▶ What type of cloud is typically associated with thunderstorms? 
  esperado: The type of cloud typically associated with thunderstorms is cumulonimbus.
  modelo:   A type of cloud is typically associated with thunderstorms.



▶ Name the author of 'Pride and Prejudice'. 
  esperado: Jane Austen.
  modelo:   The author of 'Pride and Prejudice' is William Shakespeare.



▶ What is the periodic symbol for chlorine? 
  esperado: The periodic symbol for chlorine is Cl.
  modelo:   The periodic symbol for chlorine is CH3.



▶ Correct the punctuation in the sentence. Its time to go home.
  esperado: The corrected sentence should be: 'It's time to go home.'
  modelo:   The time to go home is 3 hours.



▶ Rewrite the sentence. The lecture was delivered in a clear manner.
  esperado: The lecture was delivered clearly.
  modelo:   The lecture was delivered in a clear manner.



In [10]:
# ✍️ Instrução da turma
print(responder(gpt2, {"instruction": "Rewrite the sentence in passive voice.", "input": "The students built a GPT."}))

The students built a GPT.


Com 124M e poucos exemplos ele aprende o **formato** (responde e para no `<|endoftext|>`), mas erra fatos — "Pride and Prejudice" virou Shakespeare. Conhecimento vem do tamanho/pré-treino, não do fine-tuning; o livro usa 355M e o dataset inteiro
(e avalia as respostas com outro LLM — `ch07/03_model-evaluation`).

## 6. Onde conseguir dados? Como construir do zero?
**Prontos:** Alpaca (52k, gerado com LLM), Dolly 15k (escrito por humanos, licença comercial), OpenAssistant (conversas),
FLAN, e milhares no Hugging Face Hub (`datasets`). Em português: traduções do Alpaca/Dolly e datasets da comunidade.

**Do zero:** é só uma lista de dicionários `instruction / input / output`:

In [11]:
meus_dados = [
    {"instruction": "Traduza para o inglês.", "input": "Bom dia, turma!", "output": "Good morning, class!"},
    {"instruction": "Classifique o sentimento.", "input": "Adorei a aula de hoje.", "output": "Positivo"},
    {"instruction": "O que é um token?", "input": "", "output": "Um pedaço de texto (palavra, subpalavra ou byte) que o modelo processa."},
]
with open(os.path.join(CHECKPOINTS, "meus_dados.json"), "w", encoding="utf-8") as f:
    json.dump(meus_dados, f, ensure_ascii=False, indent=2)
print(format_input(meus_dados[2]))
print(len(InstructionDataset(meus_dados, bpe)), "exemplos prontos para o mesmo DataLoader")

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
O que é um token?
3 exemplos prontos para o mesmo DataLoader


Fontes para montar o seu: logs de atendimento/FAQ anonimizados, documentação interna, respostas escritas por especialistas,
ou **dados sintéticos** gerados por um LLM maior e revisados por humanos (`ch07/05_dataset-generation`).
Qualidade > quantidade: o paper LIMA mostrou bons resultados com ~1.000 exemplos bem curados.
Depois disso vem o alinhamento por preferência (RLHF/DPO — `ch07/04_preference-tuning-with-dpo`).

### ✅ Construto
GPT-2 pré-treinado + dados de instrução + o mesmo loop de treino = um (mini) assistente.